## Import dependencies

In [73]:
import pandas as pd
import requests
import json

import time
import random
import traceback

## Scrape all legislature names and districts to dataframe

The code in this section scrapes all legislators who have served in the Massachusetts state goverment between the years 2001 and 2024. This is done by accessing the LegislativeMember endpoint on the [MA Legislature Swagger UI webpage](https://malegislature.gov/api/swagger/index.html?url=/api/swagger/v1/swagger.json#/LegislativeMembers) and parsing the resulting items to access data for each recorded member.

Some simple exploratory data analysis of this data is included in the [following section](#eda)

In [74]:
# Define the years of the General Court stored on the Mass Legislature website
court_numbers = [i for i in range(182, 195)]
court_years =  [i for i in range(2001, 2026, 2)]

# There should be an associated court number for each court year
assert(len(court_years) == len(court_numbers))

In [75]:
# Define global header to be used in all requests
header = {"accept": "application/json"}

In [81]:
def scrape_representatives(add_delay=False):
    try:
        # Create a log file for handling parsing errors
        f = open('log.txt', 'w')

        # Create blank data frame to store information
        legislator_data = []

        for i, court_number in enumerate(court_numbers):
            print(f"Parsing data for the {court_number}th General Court...")
            f.write(f"Parsing data for the {court_number}th General Court...")
            url = f"https://malegislature.gov/api/GeneralCourts/{court_number}/LegislativeMembers"

            # Add a random delay
            if add_delay:
                time.sleep(random.uniform(1,5))

            # Rotate through proxies
            current_proxy = proxies[i % len(proxies)]

            # Get the response from the API
            resp = requests.get(url, headers=header)

            # Make sure response code is correct
            if resp.status_code == 200:

                # Get JSON from response
                data = resp.json()

                # Iterate over each representative returned
                for j, item in enumerate(data):
                    # Output parsing record number
                    print(f"Parsing record [{j} / {len(data)}] for current court")

                    # Get link to xml file
                    xml_url = item["Details"]

                    # Check for broken links containing
                    if '&' in xml_url:
                        f.write(f"Found and skipped invalid url {xml_url} in {court_number}th General Court")
                        continue

                    # Add a random delay if add_delay is true
                    if add_delay:
                        time.sleep(random.uniform(1,5))

                    xml_resp = requests.get(xml_url)

                    f.write(f"[{j}\/{len(data)}] Parsing: {xml_url}\n")

                    # Parse bytes returned into JSON and extract desired fields
                    decode_str = xml_resp.content.decode('utf-8')

                    # Check if response got a broken site
                    if '<!DOCTYPE html>' in decode_str:
                        print(f"Issue encountered and skipping processing URL: {xml_url}")

                    # Save data to list to be appended to overall list
                    legislator_info = json.loads(decode_str)
                    current_data = []
                    name = legislator_info['Name']
                    current_data.append(name)
                    current_data.append(legislator_info['MemberCode'])
                    current_data.append(legislator_info['Branch'])
                    current_data.append(legislator_info['Party'])
                    current_data.append(legislator_info['District'])
                    current_data.append(court_years[i])
                    current_data.append(court_number)

                    f.write(f"Found legislator: {name}")

                    legislator_data.append(current_data)

            else:
                print(f"Court Number {court_number} failed to parse!")
                break

        print(f"Finished parsing General Courts {min(court_numbers)} - {max(court_numbers)}! Found {len(legislator_data)} total records (not unique)")
        return legislator_data
    except:
        f.write("Error found in parsing")
        f.close()
        traceback.print_exc()


In [82]:
rep_data = scrape_representatives()

Parsing data for the 182th General Court...
Parsing record [0 / 215] for current court
Parsing record [1 / 215] for current court
Parsing record [2 / 215] for current court
Parsing record [3 / 215] for current court
Parsing record [4 / 215] for current court
Parsing record [5 / 215] for current court
Parsing record [6 / 215] for current court
Parsing record [7 / 215] for current court
Parsing record [8 / 215] for current court
Parsing record [9 / 215] for current court
Parsing record [10 / 215] for current court
Parsing record [11 / 215] for current court
Parsing record [12 / 215] for current court
Parsing record [13 / 215] for current court
Parsing record [14 / 215] for current court
Parsing record [15 / 215] for current court
Parsing record [16 / 215] for current court
Parsing record [17 / 215] for current court
Parsing record [18 / 215] for current court
Parsing record [19 / 215] for current court
Parsing record [20 / 215] for current court
Parsing record [21 / 215] for current cour

In [83]:
rep_data[0]

['Marie P. St. Fleur', 'MPS1', 'House', 'Democrat', None, 2001, 182]

In [84]:
# Turn list of representative data into a data frame
df = pd.DataFrame(rep_data, columns=['full_name', 'member_code', 'branch', 'party_affiliation',  
                                     'district', 'court_year', 'court_number'])

df.head()

,full_name,member_code,branch,party_affiliation,district,court_year,court_number
0,Marie P. St. Fleur,MPS1,House,Democrat,None,2001,182
1,Michael R. Knapik,MRK0,Senate,Republican,None,2001,182
2,Marc R. Pacheco,MRP0,Senate,Democrat,None,2001,182
3,Mary S. Rogeness,MSR1,House,Republican,None,2001,182
4,Mark V. Falzone,MVF1,House,Democrat,None,2001,182


### <a name="eda"></a>Simple EDA for the MA Representative data set

This data set is auxillary to the main data (House and Senate vote data) that will be used to supplement the project with information for linking bills to representatives. As such, the data analysis of this set is less critical to the project's outcome.

However, there are a few interesting insights that can be seen from playing with it.

In [71]:
# Calculate how many unique representatives there were out of all of the total spots possible
len(df.full_name.unique())
len(df)


214

In [ ]:
len(df.drop_duplicates(subset=['full_name']))

In [85]:
# Write the data set of representatives to a csv in the /data folder
df.to_csv('../data/MAlegislators_GC182-194.csv')